<a href="https://colab.research.google.com/github/SattikBhaumik/CS156/blob/main/CS156_Session_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Networks 1: Feed-Forward Neural Networks

##Code Cell 1

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.losses import BinaryCrossentropy

(x_train, y_train), (x_val, y_val) = mnist.load_data()

x_train = x_train.astype('float32') / 255
x_val = x_val.astype('float32') / 255 # Normalize x_val as well
y_train = to_categorical(y_train)
y_val = to_categorical(y_val) # Convert y_val to categorical as well

model = tf.keras.Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='sigmoid'),
    Dense(64, activation='sigmoid'),
    Dense(10) # Using linear activation by default for the last layer
])

model.compile(optimizer='SGD',
              loss=BinaryCrossentropy(),
              metrics=['accuracy'])

# The first fit call is redundant if validation data is provided in the second call.
# I will remove the first model.fit without validation data to avoid re-training.
# If you intend to train without validation and then with validation, keep both.
model.fit(x_train, y_train, epochs=10)

model.fit(x_train, y_train, epochs=10, validation_data=(x_val, y_val))

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.2563 - loss: 0.9606
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.1999 - loss: 2.2413
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.0992 - loss: 4.7618
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.0992 - loss: 5.3241
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.0992 - loss: 5.3223
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.0992 - loss: 5.3196
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.0992 - loss: 5.3183
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.0992 - loss: 6.7093
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.0992 - loss: 6.7343
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.0992 - loss: 6.7343
Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.0992 - loss: 6.7343 - val_accuracy: 0.1009 - val_lo

##Code Cell 2

In [ ]:
#retrieved from https://machinelearningmastery.com/implement-perceptron-algorithm-scratch-python/
#Jason Brownlee

# Perceptron Algorithm on the Sonar Dataset
from random import seed
from random import randrange
from csv import reader

# Load a CSV file
def load_csv(filename):
	dataset = list()
	with open(filename, 'r') as file:
		csv_reader = reader(file)
		for row in csv_reader:
			if not row:
				continue
			dataset.append(row)
	return dataset

# Convert string column to float
def str_column_to_float(dataset, column):
	for row in dataset:
		row[column] = float(row[column].strip())

# Convert string column to integer
def str_column_to_int(dataset, column):
	class_values = [row[column] for row in dataset]
	unique = set(class_values)
	lookup = dict()
	for i, value in enumerate(unique):
		lookup[value] = i
	for row in dataset:
		row[column] = lookup[row[column]]
	return lookup

# Split a dataset into k folds
def cross_validation_split(dataset, n_folds):
	dataset_split = list()
	dataset_copy = list(dataset)
	fold_size = int(len(dataset) / n_folds)
	for i in range(n_folds):
		fold = list()
		while len(fold) < fold_size:
			index = randrange(len(dataset_copy))
			fold.append(dataset_copy.pop(index))
		dataset_split.append(fold)
	return dataset_split

# Calculate accuracy percentage
def accuracy_metric(actual, predicted):
	correct = 0
	for i in range(len(actual)):
		if actual[i] == predicted[i]:
			correct += 1
	return correct / float(len(actual)) * 100.0

# Evaluate an algorithm using a cross validation split
def evaluate_algorithm(dataset, algorithm, n_folds, *args):
	folds = cross_validation_split(dataset, n_folds)
	scores = list()
	for fold in folds:
		train_set = list(folds)
		train_set.remove(fold)
		train_set = sum(train_set, [])
		test_set = list()
		for row in fold:
			row_copy = list(row)
			test_set.append(row_copy)
			row_copy[-1] = None
		predicted = algorithm(train_set, test_set, *args)
		actual = [row[-1] for row in fold]
		accuracy = accuracy_metric(actual, predicted)
		scores.append(accuracy)
	return scores

# Make a prediction with weights
def predict(row, weights):
	activation = weights[0]
	for i in range(len(row)-1):
		activation += weights[i + 1] * row[i]
	return 1.0 if activation >= 0.0 else 0.0

# Estimate Perceptron weights using stochastic gradient descent
def train_weights(train, l_rate, n_epoch):
	weights = [0.0 for i in range(len(train[0]))]
	for epoch in range(n_epoch):
		for row in train:
			prediction = predict(row, weights)
			error = row[-1] - prediction
			weights[0] = weights[0] + l_rate * error
			for i in range(len(row)-1):
				weights[i + 1] = weights[i + 1] + l_rate * error * row[i]
	return weights

# Perceptron Algorithm With Stochastic Gradient Descent
def perceptron(train, test, l_rate, n_epoch):
	predictions = list()
	weights = train_weights(train, l_rate, n_epoch)
	for row in test:
		prediction = predict(row, weights)
		predictions.append(prediction)
	return(predictions)

# Test the Perceptron algorithm on the sonar dataset
seed(1)
# load and prepare data
# download data from https://machinelearningmastery.com/implement-perceptron-algorithm-scratch-python/
filename = '/content/sonar.all-data'
dataset = load_csv(filename)
for i in range(len(dataset[0])-1):
	str_column_to_float(dataset, i)
# convert string class to integers
str_column_to_int(dataset, len(dataset[0])-1)
# evaluate algorithm
n_folds = 3
l_rate = 0.01
n_epoch = 500
scores = evaluate_algorithm(dataset, perceptron, n_folds, l_rate, n_epoch)
print('Scores: %s' % scores)
print('Mean Accuracy: %.3f%%' % (sum(scores)/float(len(scores))))

Scores: [76.81159420289855, 69.56521739130434, 72.46376811594203]
Mean Accuracy: 72.947%


##Extension Question

In [ ]:
# Fashion-MNIST Perceptron
#(using Jason Brownlee's perceptron algo from scratch)

import kagglehub
import pandas as pd
import numpy as np

from random import seed, randrange
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

path = kagglehub.dataset_download("zalando-research/fashionmnist")
print("Path to dataset files:", path)
filename = path + "/fashion-mnist_train.csv"

df = pd.read_csv(filename)
print("Original dataset shape:", df.shape)
print(df.head())

Using Colab cache for faster access to the 'fashionmnist' dataset.
Path to dataset files: /kaggle/input/fashionmnist
Original dataset shape: (60000, 785)
   label  pixel1  pixel2  pixel3  pixel4  pixel5  pixel6  pixel7  pixel8  \
0      2       0       0       0       0       0       0       0       0   
1      9       0       0       0       0       0       0       0       0   
2      6       0       0       0       0       0       0       0       5   
3      0       0       0       0       1       2       0       0       0   
4      3       0       0       0       0       0       0       0       0   

   pixel9  ...  pixel775  pixel776  pixel777  pixel778  pixel779  pixel780  \
0       0  ...         0         0         0         0         0         0   
1       0  ...         0         0         0         0         0         0   
2       0  ...         0         0         0        30        43         0   
3       0  ...         3         0         0         0         0         1   

In [ ]:
# selection of two classes
# Fashion-MNIST labels:
# 0 = T-shirt/top
# 1 = Trouser
# 2 = Pullover
# 3 = Dress
# 4 = Coat
# 5 = Sandal
# 6 = Shirt
# 7 = Sneaker
# 8 = Bag
# 9 = Ankle boot

# In this code run, I will classify:
# 0 -> T-shirt/top
# 1 -> Trouser

df = df[df["label"].isin([0, 1])].copy()
print("Binary dataset shape:", df.shape)
print("\nClass counts:")
print(df["label"].value_counts())

Binary dataset shape: (12000, 785)

Class counts:
label
0    6000
1    6000
Name: count, dtype: int64


In [ ]:
# getting pixels from labels
# Here, each FashionMNIST image is a 28 x 28 pixel grid
# meaning, 784 pixel features
# where each pixel value ranges from 0 to 255

X = df.drop("label", axis=1).values
y = df["label"].values


# normalising pixel values:[0, 255] to [0, 1].
X = X / 255.0

# standardising features
# PCA works better with comparable scaling features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA application
# Here, we reduce the image to 20 principal components from 784 pixels
# PCA finds directions in the image data that explain large amounts of variance
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)
print("Shape before PCA:", X.shape)
print("Shape after PCA:", X_pca.shape)
print("Variance explained by 20 components:", pca.explained_variance_ratio_.sum())

Shape before PCA: (12000, 784)
Shape after PCA: (12000, 20)
Variance explained by 20 components: 0.6992150904926512


In [ ]:
# converting data into the format used in the original perceptron code.
# Brownlee's code needs rows of the form: [feature1, feature2, ..., featureN, label]
# so I am adding y array in the end
dataset = np.column_stack((X_pca, y)).tolist()
print("Example row length:", len(dataset[0]))
print("Number of PCA features:", len(dataset[0]) - 1)
print("Label:", dataset[0][-1])

Example row length: 21
Number of PCA features: 20
Label: 0.0


In [ ]:
# given perceptron code from scratch
# dividing a dataset into k folds
def cross_validation_split(dataset, n_folds):
    dataset_split = []
    dataset_copy = list(dataset)
    fold_size = int(len(dataset) / n_folds)
    for i in range(n_folds):
        fold = []
        while len(fold) < fold_size:
            index = randrange(len(dataset_copy))
            fold.append(dataset_copy.pop(index))
        dataset_split.append(fold)
    return dataset_split


# calculating accuracy percentage
def accuracy_metric(actual, predicted):
    correct = 0
    for i in range(len(actual)):
        if actual[i] == predicted[i]:
            correct += 1
    return correct / float(len(actual)) * 100.0

# evaluation of algorithm using cross validation
def evaluate_algorithm(dataset, algorithm, n_folds, *args):
    folds = cross_validation_split(dataset, n_folds)
    scores = []
    for fold in folds:
        train_set = list(folds)
        train_set.remove(fold)
        train_set = sum(train_set, [])
        test_set = []
        for row in fold:
            row_copy = list(row)
            test_set.append(row_copy)
            # Hide the true class label from the model
            row_copy[-1] = None
        predicted = algorithm(train_set,test_set,*args)
        actual = [row[-1] for row in fold]
        accuracy = accuracy_metric(actual,predicted)
        scores.append(accuracy)
    return scores


# code for prediction
# Perceptron activation: z = b + w1*x1 + w2*x2 + ... + wn*xn
# Predict class 1 if z >= 0
# Predict class 0 otherwise
def predict(row, weights):
    activation = weights[0]   # bias
    for i in range(len(row) - 1):
        activation += weights[i + 1] * row[i]
    return 1.0 if activation >= 0.0 else 0.0


# training of perceptron weights
# Update rule: w_j <- w_j + learning_rate * error * x_j
# where:
# error = actual - predicted
def train_weights(train, l_rate, n_epoch):
    # One bias weight + one weight per PCA feature
    weights = [0.0 for i in range(len(train[0]))]
    for epoch in range(n_epoch):
        for row in train:
            prediction = predict(row, weights)
            error = row[-1] - prediction
            # Update bias
            weights[0] = (weights[0]+ l_rate * error)
            # Update feature weights
            for i in range(len(row) - 1):
                weights[i + 1] = (weights[i + 1]+ l_rate* error* row[i])
    return weights


# perceptron algorithm
def perceptron(train, test, l_rate, n_epoch):
    predictions = []
    weights = train_weights(train,l_rate,n_epoch)
    for row in test:
        prediction = predict(row,weights)
        predictions.append(prediction)
    return predictions

In [ ]:
# model evaluation
seed(1)
n_folds = 3
l_rate = 0.01
n_epoch = 100
scores = evaluate_algorithm(dataset,perceptron,n_folds,l_rate,n_epoch)

print("\nScores:", scores)
print("Mean Accuracy: %.3f%%"%(sum(scores) / len(scores)))


Scores: [97.75, 96.475, 94.8]
Mean Accuracy: 96.342%
